## SQL JOINS + ADVANCED QUERIES
### GOAL

👉 Today you will learn:
- What is a JOIN
- Types of JOINs
- How to combine multiple tables
- Solve real-world problems

### 1. WHY DO WE NEED JOINS?

👉 In real systems, data is NOT in one table

Example:

### Table 1: customers

| customer_id | name  | city    |
|-------------|-------|---------|
| 1           | Ali   | Lahore  |
| 2           | Sara  | Karachi |

### Table 2: orders

| order_id | customer_id | product | amount |
|----------|-------------|---------|--------|
| 101      | 1           | Laptop  | 100000 |
| 102      | 2           | Mobile  | 50000  |

👉 Problem:
We want:

👉 “Which customer bought what?”

👉 But data is in 2 tables

## 2. INNER JOIN (MOST IMPORTANT)

### Syntax

```sql
SELECT customers.name, orders.product
FROM customers
INNER JOIN orders
ON customers.customer_id = orders.customer_id;```

# RESULT

| name    | product |
|---------|---------|
| Ali     | Laptop  |
| Sara    | Mobile  |

👉 Meaning:
✔ Only matching data

# 3. LEFT JOIN

Shows ALL customers even if no order

## SQL

```sql
SELECT customers.name, orders.product  
FROM customers  
LEFT JOIN orders  
ON customers.customer_id = orders.customer_id;
👉 If no order → NULL

# 4. RIGHT JOIN

- Shows ALL orders even if customer missing

## SQL

```sql
SELECT customers.name, orders.product
FROM customers
RIGHT JOIN orders
ON customers.customer_id = orders.customer_id;

### 5. FULL JOIN
👉 Shows ALL data
```SELECT customers.name, orders.product
FROM customers
FULL JOIN orders
ON customers.customer_id = orders.customer_id;

### 6. REAL WORLD EXAMPLE (IMPORTANT)
👉 Imagine:

- Users table
- Jobs table
- Applications table

👉 JOIN helps:

✔ Match user → job

✔ Build recommendation systems

### 7. AGGREGATION + JOIN (POWERFUL)
👉 Find:

“Total spending per customer”
```SELECT customers.name, SUM(orders.amount) AS total_spent
FROM customers
INNER JOIN orders
ON customers.customer_id = orders.customer_id
GROUP BY customers.name;

### 8. SUBQUERY (ADVANCED)
👉 Find highest spender:
```SELECT name
FROM customers
WHERE customer_id = (
    SELECT customer_id
    FROM orders
    GROUP BY customer_id
    ORDER BY SUM(amount) DESC
    LIMIT 1
);

## SQL JOINS PROJECT
STEP 1 — Import Libraries

In [2]:
import sqlite3
import pandas as pd

### STEP 2 — Create Database

In [3]:
conn = sqlite3.connect("company.db")
cursor = conn.cursor()

This creates a mini database inside Colab.
### STEP 3 — Create Tables
**Customers Table**

In [4]:
cursor.execute("""
CREATE TABLE customers(
    customer_id INTEGER,
    name TEXT,
    city TEXT
)
""")

**Orders Table**

In [5]:
cursor.execute("""
CREATE TABLE orders(
    order_id INTEGER,
    customer_id INTEGER,
    product TEXT,
    amount INTEGER
)
""")

### STEP 4 — Insert Sample Data

In [6]:
cursor.executemany("INSERT INTO customers VALUES (?,?,?)", [
(1,"Ali","Lahore"),
(2,"Sara","Karachi"),
(3,"Ahmed","Islamabad"),   # this customer has NO orders
])

cursor.executemany("INSERT INTO orders VALUES (?,?,?,?)", [
(101,1,"Laptop",100000),
(102,2,"Mobile",50000),
(103,1,"Tablet",30000)
])

conn.commit()

Now we have two tables with real data.
### VIEW TABLES (Optional)

In [7]:
pd.read_sql_query("SELECT * FROM customers", conn)
pd.read_sql_query("SELECT * FROM orders", conn)

,order_id,customer_id,product,amount
0,101,1,Laptop,100000
1,102,2,Mobile,50000
2,103,1,Tablet,30000


### TASK 1 — JOIN customers + orders

👉 Show: name, product

INNER JOIN (only matching rows)

In [8]:
query = """
SELECT customers.name, orders.product
FROM customers
INNER JOIN orders
ON customers.customer_id = orders.customer_id
"""
pd.read_sql_query(query, conn)

,name,product
0,Ali,Laptop
1,Ali,Tablet
2,Sara,Mobile


**Explanation**
- Match both tables using customer_id
- Show who bought what

### TASK 2 — Total spending per customer

In [9]:
query = """
SELECT customers.name, SUM(orders.amount) AS total_spent
FROM customers
INNER JOIN orders
ON customers.customer_id = orders.customer_id
GROUP BY customers.name
"""
pd.read_sql_query(query, conn)

,name,total_spent
0,Ali,130000
1,Sara,50000


**Explanation**
- JOIN tables
- GROUP BY customer
- SUM their spending

This is a real business query.

### TASK 3 — Customer who spent the MOST (Subquery)

In [10]:
query = """
SELECT name
FROM customers
WHERE customer_id = (
    SELECT customer_id
    FROM orders
    GROUP BY customer_id
    ORDER BY SUM(amount) DESC
    LIMIT 1
)
"""
pd.read_sql_query(query, conn)

,name
0,Ali


**Explanation**
- Inner query finds highest spender ID
- Outer query finds their name

This is advanced SQL

### TASK 4 — Customers who did NOT place any order

👉 Use LEFT JOIN + NULL

In [11]:
query = """
SELECT customers.name
FROM customers
LEFT JOIN orders
ON customers.customer_id = orders.customer_id
WHERE orders.customer_id IS NULL
"""
pd.read_sql_query(query, conn)

,name
0,Ahmed


**Explanation**

- LEFT JOIN keeps all customers
- Customers without orders → become NULL
- We filter those NULL rows.

# WHAT WE LEARNED

| Concept              | Real meaning                       |
|----------------------|------------------------------------|
| INNER JOIN           | Only matching rows                 |
| LEFT JOIN            | Keep all left table rows           |
| GROUP BY + JOIN      | Business analytics                 |
| Subquery             | Query inside query                 |
| NULL filtering       | Find missing relationships         |

## What is JOIN?

- A JOIN combines data from multiple tables using a common column.

### INNER vs LEFT JOIN

| INNER JOIN                    | LEFT JOIN                           |
|-------------------------------|-------------------------------------|
| Only matching rows            | All left rows + matched rows        |